# 01 ? EM2023 Source Inventory

## Objective
Catalog the local source files and preserve source headers exactly as stored. This inventory reads file metadata and CSV headers only; it does not inspect microdata rows or modify files under `data/sources/raw/`.

## Document hierarchy
Official EM2023 materials (technical sheet, questionnaires, and cartilla) are the primary references for survey definitions. Contract 2022-1972 deliverables provide methodological context. Record a field definition only when a source explicitly supports its correspondence to a CSV column.

## Relevant contract materials
The Stage II methodological recommendations and thematic variables annexes provide survey-design and analysis-scope context, not a source-column dictionary. The Stage V OD-matrix methodology is reserved for later OD work. Duplicate Stage VI copies are catalogued but are not independent evidence.

## Requirements
Run `00_project_setup.ipynb` and install document-reading dependencies with `uv sync --extra source-docs`.

In [1]:
from pathlib import Path
import json
import pandas as pd

candidates = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next((path for path in candidates if (path / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not find pyproject.toml in the current directory or its parents.")

from mode_choice.source_inventory import write_catalogs

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw source directory: {PROJECT_ROOT / 'data/sources/raw'}")

Project root: D:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect
Raw source directory: D:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect\data\sources\raw


In [2]:
catalog_path, schema_path = write_catalogs(PROJECT_ROOT)
catalog = pd.read_csv(catalog_path, encoding="utf-8-sig")
schemas = json.loads(schema_path.read_text(encoding="utf-8"))

print(f"Inventoried {len(catalog)} source files.")
print(f"Catalog: {catalog_path.relative_to(PROJECT_ROOT)}")
print(f"Preliminary schema: {schema_path.relative_to(PROJECT_ROOT)}")
display(catalog.groupby("category", dropna=False).agg(files=("source_path", "count"), total_size_bytes=("size_bytes", "sum")).sort_index())

Inventoried 50 source files.
Catalog: docs\data_dictionary\source_inventory.csv
Preliminary schema: docs\data_dictionary\source_schema.json


,files,total_size_bytes
category,,
contract_context,18,51147649
em2023_documentation,5,57015665
em2023_microdata,5,165077421
geographic_reference,16,21415448
survey_instruments,3,1515309
survey_technical_documentation,3,1732618


# CSV source schemas

This view reports source headers exactly as stored. It does not show or sample records.

In [3]:
csv_catalog = catalog.loc[catalog["format"].eq("csv"), ["source_path", "csv_encoding", "csv_delimiter", "column_count", "headers_json", "empty_header_positions_json", "whitespace_header_names_json", "duplicate_headers_json", "metadata_status"]].copy()
display(csv_catalog)

# Names suggest possible identifiers, but no values or relationships are validated here.
candidate_summary = pd.DataFrame([
    {"source_path": table["source_path"], "candidate_identifiers_unverified": table["candidate_identifiers_unverified"], "status": table["candidate_status"]}
    for table in schemas
])
display(candidate_summary)

# Safe checks for expected files and metadata completeness.
expected_tables = {
    "data/sources/raw/Database/a. Modulo hogares.csv",
    "data/sources/raw/Database/b. Modulo vehiculos.csv",
    "data/sources/raw/Database/c. Modulo personas.csv",
    "data/sources/raw/Database/d. Modulo viajes.csv",
    "data/sources/raw/Database/e. Modulo etapas.csv",
}
found_tables = set(csv_catalog["source_path"])
assert expected_tables.issubset(found_tables), f"Missing expected source files: {sorted(expected_tables - found_tables)}"
assert csv_catalog.loc[csv_catalog["source_path"].isin(expected_tables), "metadata_status"].eq("ok").all()

,source_path,csv_encoding,csv_delimiter,column_count,headers_json,empty_header_positions_json,whitespace_header_names_json,duplicate_headers_json,metadata_status
18,data/sources/raw/Database/a. Modulo hogares.csv,cp1252,;,27.0,"[""fecha"", ""cod_viv"", ""cod_hog"", ""cod_utam_hg"",...",[],[],[],ok
19,data/sources/raw/Database/b. Modulo vehiculos.csv,cp1252,;,19.0,"[""cod_vh"", ""cod_viv"", ""cod_hog"", ""cod_utam_hg""...",[19],[],[],ok
20,data/sources/raw/Database/c. Modulo personas.csv,cp1252,;,40.0,"[""cod_per"", ""cod_hg"", ""nom_mun_hg"", ""cod_upl_h...",[],[],[],ok
21,data/sources/raw/Database/d. Modulo viajes.csv,cp1252,;,47.0,"[""cod_hg"", ""nom_mun_hg"", ""cod_upl_hg"", ""cod_ut...",[],"["" hora_ini "", "" duracion_min ""]",[],ok
22,data/sources/raw/Database/e. Modulo etapas.csv,cp1252,;,38.0,"[""cod_eta"", ""cod_vj"", ""cod_pers"", ""cod_hg"", ""n...",[],[],[],ok


,source_path,candidate_identifiers_unverified,status
0,data/sources/raw/Database/a. Modulo hogares.csv,"[cod_viv, cod_hog, key_hg]",name_based_candidates_only; uniqueness_and_sco...
1,data/sources/raw/Database/b. Modulo vehiculos.csv,"[cod_vh, cod_viv, cod_hog, key_hg]",name_based_candidates_only; uniqueness_and_sco...
2,data/sources/raw/Database/c. Modulo personas.csv,"[cod_per, key_hg, key_persona]",name_based_candidates_only; uniqueness_and_sco...
3,data/sources/raw/Database/d. Modulo viajes.csv,"[cod_pers, cod_vj, key_hg, key_pers, key_pers_...",name_based_candidates_only; uniqueness_and_sco...
4,data/sources/raw/Database/e. Modulo etapas.csv,"[cod_eta, cod_vj, cod_pers, key_viaje, key_etapa]",name_based_candidates_only; uniqueness_and_sco...


## Document review notes

- The EM2023 technical survey sheet describes survey design and sample coverage. Use it to frame the survey universe and later weight review, not as a variable dictionary.
- The household and travel forms describe questionnaire modules and concepts. A form question is not automatically a one-to-one definition of a CSV field; record a field definition only when its correspondence is explicit.
- The EM2023 cartilla is an analytical and results publication. It provides context, but its aggregate indicators do not establish source-table keys or field-level definitions.
- The Stage II Annex 3 workbook summarizes methodological recommendations, including study area, sample design, and form structure. Annex 4 organizes topics and variables of interest; its thematic codes still need mapping to source fields.
- The Stage IV final report provides context on field validation and EODI georeferencing. The Stage V OD methodology is relevant to later Level 1 OD work, not a direct specification for the relational layer or a mode-choice dataset.
- The 2019 sample-distribution workbook is historical context only. Two Stage VI deliverables are byte-identical copies of the technical sheet and indicator report already catalogued elsewhere.
- Indicator workbooks are catalogued by workbook and sheet metadata. Treat published indicators as comparison references, not as microdata schemas.

Detailed source roles and review references are recorded in `docs/data_dictionary/source_review.md`.